In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/mental_health.sqlite')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

       name
0    Answer
1  Question
2    Survey


In [2]:
df_answer = pd.read_sql("SELECT * FROM Answer", conn)
df_question = pd.read_sql("SELECT * FROM Question", conn)
df_survey = pd.read_sql("SELECT * FROM Survey", conn)

df = pd.read_sql("""
    SELECT a.UserID, a.SurveyID, a.AnswerText, q.questiontext
    FROM Answer a
    JOIN Question q ON a.QuestionID = q.questionid
""", conn)

df.head(10)

,UserID,SurveyID,AnswerText,questiontext
0,1,2014,37,What is your age?
1,2,2014,44,What is your age?
2,3,2014,32,What is your age?
3,4,2014,31,What is your age?
4,5,2014,31,What is your age?
5,6,2014,33,What is your age?
6,7,2014,35,What is your age?
7,8,2014,39,What is your age?
8,9,2014,42,What is your age?
9,10,2014,23,What is your age?


In [3]:
print(df['questiontext'].unique())
print(df['SurveyID'].unique())

['What is your age?' 'What is your gender?' 'What country do you live in?'
 'If you live in the United States, which state or territory do you live in?'
 'Are you self-employed?'
 'Do you have a family history of mental illness?'
 'Have you ever sought treatment for a mental health disorder from a mental health professional?'
 'How many employees does your company or organization have?'
 'Is your employer primarily a tech company/organization?'
 'Does your employer provide mental health benefits as part of healthcare coverage?'
 'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?'
 'Would you bring up a mental health issue with a potential employer in an interview?'
 'Do you think that discussing a physical health issue with your employer would have negative consequences?'
 'Do you feel that your employer takes mental health as seriously as physical health?'
 'If you have a mental health conditi

In [6]:
treatment = df[df['questiontext'] == 'Have you ever sought treatment for a mental health disorder from a mental health professional?']
print(treatment['AnswerText'].value_counts())

AnswerText
1    2412
0    1806
Name: count, dtype: int64


In [7]:
df_wide = df.pivot_table(
    index=['UserID', 'SurveyID'],
    columns='questiontext',
    values='AnswerText',
    aggfunc='first'
).reset_index()

print(df_wide.shape)
df_wide.head()

(4218, 107)


questiontext,UserID,SurveyID,Any additional notes or comments,Are you openly identified at work as a person with a mental health issue?,Are you self-employed?,Briefly describe what you think the industry as a whole and/or employers could do to improve mental health support for employees.,Describe the circumstances of the badly handled or unsupportive response.,Describe the circumstances of the supportive or well handled response.,Describe the conversation with coworkers you had about your mental health including their reactions.,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",...,Would you be willing to discuss a mental health issue with your direct supervisor(s)?,Would you be willing to talk to one of us more extensively about your experiences with mental health issues in the tech industry? (Note that all interview responses would be used _anonymously_ and only with your permission.),Would you bring up a mental health issue with a potential employer in an interview?,Would you bring up a physical health issue with a potential employer in an interview?,Would you feel comfortable discussing a mental health issue with your coworkers?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you have been willing to discuss a mental health issue with your previous co-workers?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?
0,1,2014,-1,NaN,-1,NaN,NaN,NaN,NaN,NaN,...,Yes,NaN,No,Maybe,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2014,-1,NaN,-1,NaN,NaN,NaN,NaN,NaN,...,No,NaN,No,No,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2014,-1,NaN,-1,NaN,NaN,NaN,NaN,NaN,...,Yes,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2014,-1,NaN,-1,NaN,NaN,NaN,NaN,NaN,...,No,NaN,Maybe,Maybe,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2014,-1,NaN,-1,NaN,NaN,NaN,NaN,NaN,...,Yes,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print(df_wide.shape)

missing = (df_wide.isnull().sum() / len(df_wide) * 100).sort_values(ascending=False)
print(missing)

(4218, 107)
questiontext
Any additional notes or comments                                                                                                                  70.128023
Does your employer provide resources to learn more about mental health issues and how to seek help?                                               70.128023
Do you work remotely (outside of an office) at least 50% of the time?                                                                             70.128023
Do you know the options for mental health care your employer provides?                                                                            70.128023
Do you think that discussing a mental health issue with your employer would have negative consequences?                                           70.128023
                                                                                                                                                    ...    
Is your anonymity protected if you choo